# pH-Dependent Free Energies of Protein Association from Protonation Ensemble Statistics
Stefan Hervø-Hansen<sup>a</sup> and Nobuyuki Matubayasi<sup>a</sup><br><br>
<sup>a</sup>Division of Chemical Engineering, Graduate School of Engineering Science, The University of Osaka, Toyonaka, Osaka 560-8531, Japan.<br>
*Correspondence:* stefan@cheng.es.osaka-u.ac.jp, nobuyuki@cheng.es.osaka-u.ac.jp

---


### Module Loading

In [16]:
import os
from multiprocessing import Pool
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import (TwoSlopeNorm, LinearSegmentedColormap)
import mdtraj as md
from scipy.interpolate import RegularGridInterpolator
from IPython.display import Markdown
from scipy.interpolate import interp1d
from matplotlib.ticker import NullLocator




## Function Definitions

In [2]:
# --------------------------------------------------
# Encode residue-specific protonation rules
# --------------------------------------------------

def _encode_residue_types(resnames):
    """
    Encode residue types as integers for fast protonation checks.

    Returns
    -------
    rtype : np.ndarray
        Integer codes per residue:
        0 = GLU/ASP/GL4   (protonated if state != 0)
        1 = LYS/TYR/CYS   (protonated if state != 1)
        2 = HIS           (protonated if state == 0)
    """

    rtype = np.empty(len(resnames), dtype=np.int8)

    for i, resname in enumerate(resnames):

        if resname in ["GLU", "ASP", "GL4"]:
            rtype[i] = 0

        elif resname in ["LYS", "TYR", "CYS"]:
            rtype[i] = 1

        elif resname == "HIS":
            rtype[i] = 2

        else:
            raise ValueError(f"Unknown residue type: {resname}")

    return rtype


# --------------------------------------------------
# Fast protonation classifier (clear semantics)
# --------------------------------------------------

def _is_protonated(rtype, state):
    """
    Determine if a residue is protonated.
    """

    return (
        (rtype == 0 and state != 0) or   # GLU/ASP/GL4
        (rtype == 1 and state != 1) or   # LYS/TYR/CYS
        (rtype == 2 and state == 0)      # HIS
    )


# --------------------------------------------------
# Worker (single pH)
# --------------------------------------------------

def _process_pH(args):
    base_path, pH, rtype, fraction, stride, min_frames = args

    filename = f"{base_path}/reordered_cpouts.pH_{pH:.2f}"

    if not os.path.isfile(filename):
        return pH, None

    n_res = len(rtype)
    N_frames = []

    with open(filename, "r") as f:

        residue_counter = 0
        current_N = 0

        for line in f:

            if line.startswith("Residue"):

                parts = line.split()
                res_index = int(parts[1])
                state = int(parts[-1])

                if _is_protonated(rtype[res_index], state):
                    current_N += 1

                residue_counter += 1

                if residue_counter == n_res:
                    N_frames.append(current_N)
                    current_N = 0
                    residue_counter = 0

    # Convert to NumPy
    N = np.array(N_frames, dtype=np.int16)

    # ------------------------------
    # Trimming + thinning
    # ------------------------------
    n_total = len(N)

    if n_total == 0:
        return pH, {"N": N, "n_frames": 0}

    if fraction < 1.0:
        start = int((1 - fraction) * n_total)

        if n_total - start < min_frames:
            start = max(0, n_total - min_frames)
    else:
        start = 0

    N = N[start::stride]

    return pH, {
        "N": N,
        "n_frames": len(N)
    }


# --------------------------------------------------
# Main function (clean + expressive)
# --------------------------------------------------

def load_protonation_counts(
    base_path,
    pHs,
    resnames,
    nprocs=None,
    fraction=1.0,
    stride=1,
    min_frames=1,
    verbose=False
):
    """
    Load protonation counts N for each pH.

    Includes:
    - parallel file loading
    - residue-aware protonation logic
    - optional equilibration trimming
    - optional subsampling

    Returns
    -------
    data : dict
        {pH: {"N": array, "n_frames": int}}
    """

    if nprocs is None:
        nprocs = min(len(pHs), os.cpu_count())

    nprocs = max(1, nprocs)

    # Precompute residue encoding once
    rtype = _encode_residue_types(resnames)

    args = [
        (base_path, pH, rtype, fraction, stride, min_frames)
        for pH in pHs
    ]

    data = {}

    with Pool(processes=nprocs) as pool:
        results = pool.map(_process_pH, args)

    for pH, result in results:

        if verbose and result is None:
            print(f"Missing: pH {pH:.2f}")
            continue

        data[pH] = result
        if verbose:
            print(f"Loaded pH {pH:.2f}: {result['n_frames']} frames")

    return data

# ----------------------
# Subsampling utilities 
# ----------------------
def subsample_data(N_frames, stride):
    return np.asarray(N_frames)[::stride]


def compute_effective_sample_size(N_frames, stride):
    return len(subsample_data(N_frames, stride))


def compute_subsampled_error(N_frames, stride):
    N_sub = subsample_data(N_frames, stride)
    return np.std(N_sub, ddof=1) / np.sqrt(len(N_sub))


def compute_var_error(N_frames, stride=20):
    """
    Estimate uncertainty of Var(N) using subsampling
    """
    N_sub = N_frames[::stride]

    # variance of each block (simple approach)
    return np.std((N_sub - np.mean(N_sub))**2, ddof=1) / np.sqrt(len(N_sub))


# -----------------------------
# Observables WITH subsampling 
# -----------------------------
def extract_observables(data, pHs, stride=20):

    N_avg = []
    var_N = []
    M_eff = []
    err_N = []

    for pH in pHs:

        if pH not in data:
            raise ValueError(f"Missing pH {pH}")

        N = data[pH]["N"]

        # Mean (full data)
        N_avg.append(np.mean(N))

        # Variance (full data, best estimate)
        var_N.append(np.var(N, ddof=1))

        # Effective sample size
        M_eff_k = compute_effective_sample_size(N, stride)
        M_eff.append(M_eff_k)

        # Error (subsampled)
        err_N.append(compute_subsampled_error(N, stride))

    return (
        np.array(N_avg),
        np.array(var_N),
        np.array(M_eff),
        np.array(err_N)
    )


# -----------------------------
# Reference model
# -----------------------------
def protonation_probability(pH, pKa):
    """⟨n⟩ for a single site"""
    return 1.0 / (1.0 + 10**(pH - pKa))


def compute_N_avg_reference(pHs, pKas):
    """⟨N⟩ for non-interacting reference system"""
    pHs = np.asarray(pHs)
    pKas = np.asarray(pKas)

    probs = protonation_probability(pHs[:, None], pKas[None, :])
    return probs.sum(axis=1)

    
def compute_var_reference(pHs, pKas):
    """
    Analytical Var(N) for independent sites
    Var(N) = sum_i p_i (1 - p_i)
    """
    pHs = np.asarray(pHs)
    pKas = np.asarray(pKas)

    probs = protonation_probability(pHs[:, None], pKas[None, :])
    return np.sum(probs * (1.0 - probs), axis=1)


# -----------------------------
# Thermodynamic Integration
# -----------------------------
def compute_Upsilon_TI_profile(pHs, N_avg, pH_ref=7.0):
    """Cumulative TI curve with reference at chosen pH (Eq. 8)"""
    kBT = 2.479
    ln10 = np.log(10)

    pHs = np.asarray(pHs)
    N_avg = np.asarray(N_avg)

    Upsilon = np.zeros_like(pHs, dtype=float)

    for i in range(1, len(pHs)):
        dph = pHs[i] - pHs[i-1]
        avg = 0.5 * (N_avg[i] + N_avg[i-1])
        Upsilon[i] = Upsilon[i-1] + kBT * ln10 * avg * dph

    # Shift reference (does NOT affect variance)
    idx_ref = np.argmin(np.abs(pHs - pH_ref))
    Upsilon -= Upsilon[idx_ref]

    return Upsilon


def compute_TI_statistical_variance(pHs, var_N, M_eff):

    kBT = 2.479
    ln10 = np.log(10)

    pHs = np.asarray(pHs)
    var_N = np.asarray(var_N)
    M_eff = np.asarray(M_eff)

    dph = np.diff(pHs)

    weights = np.zeros_like(pHs, dtype=float)
    weights[0] = 0.5 * dph[0]
    weights[-1] = 0.5 * dph[-1]
    weights[1:-1] = 0.5 * (dph[:-1] + dph[1:])

    # Full TI prefactor
    weights *= (kBT * ln10)

    return np.sum((weights**2) * (var_N / M_eff))


def compute_TI_discretization_variance(pHs, var_N):
    """
    Leading-order trapezoidal discretization error estimate.
    """
    kBT = 2.479
    ln10 = np.log(10)

    pHs = np.asarray(pHs)
    var_N = np.asarray(var_N)

    dph = np.diff(pHs)

    # Finite-difference approximation to dVar(N)/dpH
    dvar = np.diff(var_N)

    return np.sum(
        ((dph**4) * (kBT**2) * (ln10**6) / 144.0)
        * (dvar**2)
    )


def compute_TI_total_variance(pHs, var_N, M_eff, verbose=False):

    var_stat = compute_TI_statistical_variance(pHs, var_N, M_eff)
    var_disc = compute_TI_discretization_variance(pHs, var_N)
    var_total = var_stat + var_disc

    # Determine dominant contribution
    if var_stat > var_disc:
        dominant = "statistical"
        ratio = var_stat / var_disc if var_disc > 0 else np.inf
    elif var_disc > var_stat:
        dominant = "discretization"
        ratio = var_disc / var_stat if var_stat > 0 else np.inf
    else:
        dominant = "equal"
        ratio = 1.0

    if verbose:
        print(f"Statistical variance     : {var_stat:.6e}")
        print(f"Discretization variance : {var_disc:.6e}")
        print(f"Total variance          : {var_total:.6e}")
        print(f"Dominant contribution   : {dominant}")

        if np.isfinite(ratio):
            print(f"Dominance ratio         : {ratio:.2f}x")
        else:
            print("Dominance ratio         : infinite")

    return var_total



def compute_deltaG_variance(var_Gn, var_G1, n):
    """
    Error propagation for:
    ΔG = G(n)/n - G(1)
    """
    return (var_Gn / (n**2)) + var_G1


## Insulin residue metadata (canonical labeling)

This cell constructs a canonical residue mapping directly from the simulation topology (`insulin_fibril_leap_output.pdb`) to ensure consistent and reproducible residue identification across all systems.

Rather than relying on hard-coded indices, residues are defined using biologically meaningful descriptors:

- chain identity
- residue sequence number (resSeq)
- residue name

From this mapping, we construct:

- residue_table: complete list of residues in the system
- titratable_residues: subset of residues included in CpHMD analysis
- titratable_indices: topology indices in the exact order used in simulation outputs

To enable direct comparison with canonical insulin, residues are aligned to the mature human insulin sequences (A and B chains). This allows automatic detection of truncations and assignment of canonical residue indices.

These objects define a consistent reference layer for all subsequent analysis and ensure that residue identities remain comparable across monomeric and protofibril systems, even when constructs differ.

In [3]:
# ============================================================
# Canonical mature human insulin
# ============================================================

INSULIN_B = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"
INSULIN_A = "GIVEQCCTSICSLYQLENYCN"

CANONICAL = {"A": INSULIN_A, "B": INSULIN_B}

CAP_RESNAMES = {"ACE","NME","NH2","CT3"}

AA3_TO_AA1 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLU":"E","GL4":"E",
    "GLN":"Q","GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M",
    "PHE":"F","PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V"
}

TITRATABLE = {"GL4","HIS","LYS","TYR"}

# ============================================================
# Load topology
# ============================================================

topology = md.load("/data9/stefan/CpHMD/fibril_1/0_preparation/insulin_fibril_leap_output.pdb")
topology.remove_solvent(inplace=True)
top = topology.topology

# ============================================================
# Build residue table
# ============================================================

residue_table = []

for res in top.residues:
    residue_table.append({
        "top_index": res.index,
        "chain": res.chain.index,
        "resSeq": res.resSeq,
        "resname": res.name,
        "oneletter": AA3_TO_AA1.get(res.name,"X"),
        "is_cap": res.name in CAP_RESNAMES
    })

# ============================================================
# Assign biological chains (A/B)
# ============================================================

chains = sorted(set(r["chain"] for r in residue_table))

chain_lengths = {
    c: sum(1 for r in residue_table if r["chain"]==c and not r["is_cap"])
    for c in chains
}

sorted_chains = sorted(chain_lengths.items(), key=lambda x:x[1])

chain_map = {
    sorted_chains[0][0]:"A",
    sorted_chains[1][0]:"B"
}

for r in residue_table:
    r["bio_chain"] = chain_map[r["chain"]]

# ============================================================
# Extract fibril sequences
# ============================================================

fibril_seq = {}

for chain in ["A","B"]:
    seq = "".join(
        r["oneletter"]
        for r in sorted(
            [x for x in residue_table if x["bio_chain"]==chain and not x["is_cap"]],
            key=lambda x:x["resSeq"]
        )
    )
    fibril_seq[chain] = seq

# ============================================================
# Detect truncations + assign canonical indices
# ============================================================

offsets = {}

print("\n===== Fibril truncation report =====\n")

for chain in ["B","A"]:

    canon = CANONICAL[chain]
    fib = fibril_seq[chain]

    start = canon.find(fib)
    if start == -1:
        raise RuntimeError(f"Could not align chain {chain}")

    offsets[chain] = start

    print(f"Chain {chain}")
    print("Detected fibril:", fib)
    print("N-cut:", canon[:start])
    print("C-cut:", canon[start+len(fib):])
    print()

    chain_res = [
        r for r in residue_table
        if r["bio_chain"]==chain and not r["is_cap"]
    ]

    chain_res.sort(key=lambda x:x["resSeq"])

    for i, r in enumerate(chain_res):
        r["canonical_index"] = i + 1 + start

# ============================================================
# Analysis-ready objects
# ============================================================

titratable_residues = [
    r for r in residue_table
    if r["resname"] in TITRATABLE and not r["is_cap"]
]

titratable_indices = [r["top_index"] for r in titratable_residues]



===== Fibril truncation report =====

Chain B
Detected fibril: HLCGSHLVEALYLVCGERGFFYT
N-cut: FVNQ
C-cut: PKT

Chain A
Detected fibril: QCCTSICSLYQLENYC
N-cut: GIVE
C-cut: N



## Loading data

In [4]:
pHs = np.array([
    0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5,
    5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5,
    10.0, 10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5
])

# Monomer
resnames_1mer = [r["resname"] for r in titratable_residues]
data_1mer = load_protonation_counts(
    base_path="/data9/stefan/CpHMD/fibril_1/5_analysis",
    pHs=pHs,
    resnames=resnames_1mer
)

# Dimer
resnames_2mer = resnames_1mer * 2
data_2mer = load_protonation_counts(
    base_path="/data9/stefan/CpHMD/fibril_2/5_analysis",
    pHs=pHs,
    resnames=resnames_2mer
)

# Trimer
resnames_3mer = resnames_1mer * 3
data_3mer = load_protonation_counts(
    base_path="/data9/stefan/CpHMD/fibril_3/5_analysis",
    pHs=pHs,
    resnames=resnames_3mer
)

# Tetramer
resnames_4mer = resnames_1mer * 4
data_4mer = load_protonation_counts(
    base_path="/data9/stefan/CpHMD/fibril_4/5_analysis",
    pHs=pHs,
    resnames=resnames_4mer
)

# Pentamer
resnames_5mer = resnames_1mer * 5
data_5mer = load_protonation_counts(
    base_path="/data9/stefan/CpHMD/fibril_5/5_analysis",
    pHs=pHs,
    resnames=resnames_5mer
)

# Hexamer
resnames_6mer = resnames_1mer * 6
data_6mer = load_protonation_counts(
    base_path="/data9/stefan/CpHMD/fibril_6/5_analysis",
    pHs=pHs,
    resnames=resnames_6mer
)

# Heptamer
resnames_7mer = resnames_1mer * 7
data_7mer = load_protonation_counts(
    base_path="/data9/stefan/CpHMD/fibril_7/5_analysis",
    pHs=pHs,
    resnames=resnames_7mer
)

# Octamer
resnames_8mer = resnames_1mer * 8
data_8mer = load_protonation_counts(
    base_path="/data9/stefan/CpHMD/fibril_8/5_analysis",
    pHs=pHs,
    resnames=resnames_8mer
)


## From Macroscopic Titration Behavior to pH-Dependent Free Energies
### Extracting observables and subsampling

In [5]:
N_1mer, var_1mer, M_1mer, err_1mer = extract_observables(data_1mer, pHs)
N_2mer, var_2mer, M_2mer, err_2mer = extract_observables(data_2mer, pHs)
N_3mer, var_3mer, M_3mer, err_3mer = extract_observables(data_3mer, pHs)
N_4mer, var_4mer, M_4mer, err_4mer = extract_observables(data_4mer, pHs)
N_5mer, var_5mer, M_5mer, err_5mer = extract_observables(data_5mer, pHs)
N_6mer, var_6mer, M_6mer, err_6mer = extract_observables(data_6mer, pHs)
N_7mer, var_7mer, M_7mer, err_7mer = extract_observables(data_7mer, pHs)
N_8mer, var_8mer, M_8mer, err_8mer = extract_observables(data_8mer, pHs)


### ⟨N⟩ vs. pH

In [13]:
with plt.rc_context({
                     'text.color': 'white',
                     'axes.labelcolor': 'white',
                     'xtick.color': 'white',
                     'ytick.color': 'white',
                     'axes.edgecolor': 'white',
                     'figure.facecolor': 'FFFFFF',
                     'axes.facecolor': 'FFFFFF',
                     'axes.linewidth': 3,
                     'font.family': 'Nimbus Sans',
                     'font.weight': 'normal',
                     'mathtext.fontset': 'stix',
                     'axes.labelweight': 'normal',
                     'axes.titleweight': 'normal',
                     'text.usetex': False,
                     'figure.figsize': (6, 4),
                     }):

    colors = ['#32ff14', '#ff7a00', '#32eefc', '#ee00ff']

    fig, ax = plt.subplots(nrows=1, ncols=1)

    ys = [N_1mer/1, N_3mer/3, N_5mer/5, N_7mer/7,]

    labels = [r"$n=1$", r"$n=3$", r"$n=5$", r"$n=7$"]

    # -------------------------
    # Neon glow settings
    # -------------------------
    n_glow = 18
    glow_alpha = 0.035
    glow_width = 1.15

    # -------------------------
    # Glow layers
    # -------------------------
    for y, color in zip(ys, colors):
        for n in range(1, n_glow+1):
            ax.plot(pHs, y, color=color, linewidth=2 + glow_width*n, alpha=glow_alpha, solid_capstyle='round', zorder=1)

    # -------------------------
    # Main foreground lines
    # -------------------------
    for y, color, label in zip(ys, colors, labels):
        ax.plot(pHs, y, linewidth=2.5, marker='o', markersize=5, color=color, label=label, solid_capstyle='round', zorder=10)

    # -------------------------
    # Axes styling
    # -------------------------
    ax.set_xlim(-0.2, 13.7)
    ax.set_xlabel("pH", fontsize=18)
    ax.set_ylabel(r'$\langle N \rangle / n$', fontsize=18)
    ax.minorticks_on()
    ax.tick_params(axis='both', which='major', labelsize=14, length=6, width=2)
    ax.tick_params(axis='both', which='minor', length=3, width=1.5)
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # -------------------------
    # Legend
    # -------------------------
    legend = ax.legend(loc='best', frameon=False, fontsize=14)

    # Make legend text white
    for text in legend.get_texts():
        text.set_color('white')

    fig.tight_layout()
    fig.savefig('TEMP_POSTER/Titration_profiles_odd.pdf', bbox_inches='tight', transparent=True)

### $\Delta \nu_n(\mathrm{pH})$ vs. pH

In [14]:
with plt.rc_context({
                     'text.color': 'white',
                     'axes.labelcolor': 'white',
                     'xtick.color': 'white',
                     'ytick.color': 'white',
                     'axes.edgecolor': 'white',
                     'figure.facecolor': 'FFFFFF',
                     'axes.facecolor': 'FFFFFF',
                     'axes.linewidth': 3,
                     'font.family': 'Nimbus Sans',
                     'font.weight': 'normal',
                     'mathtext.fontset': 'stix',
                     'axes.labelweight': 'normal',
                     'axes.titleweight': 'normal',
                     'text.usetex': False,
                     'figure.figsize': (6, 4),
                     }):

    colors = ['#32ff14', '#ff7a00', '#32eefc', '#ee00ff']

    fig, ax = plt.subplots(nrows=1, ncols=1)

    ys = [N_3mer/3 - N_1mer, N_5mer/5 - N_1mer, N_7mer/7 - N_1mer]

    labels = [r"$n=3$", r"$n=5$",r"$n=7$"]

    plot_colors = [colors[0], colors[1], colors[2]]  
    

    # -------------------------
    # Neon glow settings
    # -------------------------
    n_glow = 18
    glow_alpha = 0.035
    glow_width = 1.15

    # -------------------------
    # Glow layers
    # -------------------------
    for y, color in zip(ys, plot_colors):
        for n in range(1, n_glow+1):
            ax.plot(pHs, y, color=color, linewidth=2 + glow_width*n, alpha=glow_alpha, solid_capstyle='round', zorder=1)

    # -------------------------
    # Main foreground curves
    # -------------------------
    for y, color, label in zip(ys, plot_colors, labels):
        ax.plot(pHs, y, linewidth=2.5, marker='o', markersize=5, color=color, label=label, solid_capstyle='round', zorder=10)

    # -------------------------
    # Horizontal reference line
    # -------------------------
    ax.hlines(0, -1, 14, colors='white', alpha=0.35, linestyle='dashed', linewidth=2.0, zorder=0)

    # Soft glow for reference line
    for n in range(1, 12):
        ax.hlines(0, -1, 14, colors='white', alpha=0.015, linestyle='solid', linewidth=2 + n*1.2, zorder=0)

    # -------------------------
    # Axes styling
    # -------------------------
    ax.set_xlim(-0.2, 13.7)
    ax.set_xlabel("pH", fontsize=18)
    ax.set_ylabel(r'$\Delta \nu_n(\mathrm{pH})$', fontsize=18)
    ax.minorticks_on()
    ax.tick_params(axis='both', which='major', labelsize=14, length=6, width=2)
    ax.tick_params(axis='both', which='minor', length=3, width=1.5)
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # -------------------------
    # Legend
    # -------------------------
    legend = ax.legend(loc='best', frameon=False, fontsize=14)

    for text in legend.get_texts():
        text.set_color('white')

    fig.tight_layout()
    fig.savefig('TEMP_POSTER/Differential_protonation_response_odd.pdf', bbox_inches='tight', transparent=True)

### $\Delta \Upsilon$(pH) vs. pH

In [15]:
# Compute TI curves
Upsilon_1mer = compute_Upsilon_TI_profile(pHs, N_1mer, pH_ref=0)
Upsilon_2mer = compute_Upsilon_TI_profile(pHs, N_2mer, pH_ref=0)
Upsilon_3mer = compute_Upsilon_TI_profile(pHs, N_3mer, pH_ref=0)
Upsilon_4mer = compute_Upsilon_TI_profile(pHs, N_4mer, pH_ref=0)
Upsilon_5mer = compute_Upsilon_TI_profile(pHs, N_5mer, pH_ref=0)
Upsilon_6mer = compute_Upsilon_TI_profile(pHs, N_6mer, pH_ref=0)
Upsilon_7mer = compute_Upsilon_TI_profile(pHs, N_7mer, pH_ref=0)
Upsilon_8mer = compute_Upsilon_TI_profile(pHs, N_8mer, pH_ref=0)

# Per-monomer difference
Delta_Upsilon_2mer_per_monomer = (Upsilon_2mer / 2.0) - Upsilon_1mer
Delta_Upsilon_3mer_per_monomer = (Upsilon_3mer / 3.0) - Upsilon_1mer
Delta_Upsilon_4mer_per_monomer = (Upsilon_4mer / 4.0) - Upsilon_1mer
Delta_Upsilon_5mer_per_monomer = (Upsilon_5mer / 5.0) - Upsilon_1mer
Delta_Upsilon_6mer_per_monomer = (Upsilon_6mer / 6.0) - Upsilon_1mer
Delta_Upsilon_7mer_per_monomer = (Upsilon_7mer / 7.0) - Upsilon_1mer
Delta_Upsilon_8mer_per_monomer = (Upsilon_8mer / 8.0) - Upsilon_1mer

### Free energy landscape $\Delta\Delta\Upsilon(\mathrm{pH},n)$ 

In [17]:
# ============================================
# RC settings
# ============================================

with plt.rc_context({
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': 'white',

    # Dark neon background
    'figure.facecolor': '#FFFFFF', # #050505
    'axes.facecolor': '#FFFFFF',   # #050505

    'axes.linewidth': 3,

    # Typography
    'font.family': 'Nimbus Sans',
    'font.weight': 'normal',

    'mathtext.fontset': 'stix',

    'axes.labelweight': 'normal',
    'axes.titleweight': 'normal',

    'text.usetex': False,

    'figure.figsize': (6, 4),

}):

    # ============================================
    # Neon palette
    # ============================================

    colors = ['#32ff14', '#ff7a00', '#32eefc', '#ee00ff']

    neon_cmap = LinearSegmentedColormap.from_list(
        'neon_diverging',
        [
            '#32eefc',   # cyan
            '#050505',   # near-black midpoint
            '#ee00ff'    # magenta
        ],
        N=512
    )

    # ============================================
    # Data
    # ============================================

    n_values = np.array([1,2,3,4,5,6,7,8])

    heatmap_data = np.vstack([
        np.zeros_like(pHs),
        Delta_Upsilon_2mer_per_monomer,
        Delta_Upsilon_3mer_per_monomer,
        Delta_Upsilon_4mer_per_monomer,
        Delta_Upsilon_5mer_per_monomer,
        Delta_Upsilon_6mer_per_monomer,
        Delta_Upsilon_7mer_per_monomer,
        Delta_Upsilon_8mer_per_monomer
    ])

    # ============================================
    # Interpolate ONLY along pH
    # ============================================

    pH_fine = np.linspace(
        pHs.min(),
        pHs.max(),
        400
    )

    Z_interp = np.zeros((len(n_values), len(pH_fine)))

    for i in range(len(n_values)):

        f = interp1d(
            pHs,
            heatmap_data[i],
            kind='cubic',
            bounds_error=False,
            fill_value='extrapolate'
        )

        Z_interp[i] = f(pH_fine)

    # ============================================
    # Create discrete y-bin edges
    # ============================================

    y_edges = np.arange(0.5, 8.5 + 1, 1)

    x_edges = np.linspace(
        pH_fine.min(),
        pH_fine.max(),
        len(pH_fine)+1
    )

    # ============================================
    # Color normalization
    # ============================================

    vmax = np.nanmax(np.abs(Z_interp))

    norm = TwoSlopeNorm(
        vmin=-vmax,
        vcenter=0.0,
        vmax=vmax
    )

    # ============================================
    # Plot
    # ============================================

    fig, ax = plt.subplots()

    mesh = ax.pcolormesh(
        x_edges,
        y_edges,
        Z_interp,
        cmap=neon_cmap,
        norm=norm,
        shading='flat',
        rasterized=True
    )

    # ============================================
    # Contours
    # ============================================

    n_contour = np.append(n_values, 9)

    Z_contour = np.vstack([
        Z_interp,
        Z_interp[-1]
    ])

    P, N = np.meshgrid(
        pH_fine,
        n_contour
    )

    levels = [-20, -15, -10, -5, -2, 2, 5, 10, 20]

    # --------------------------------------------
    # Glow contours
    # --------------------------------------------

    for lw, alpha in zip(
        [10, 8, 6, 4],
        [0.015, 0.025, 0.04, 0.06]
    ):

        ax.contour(
            P,
            N,
            Z_contour,
            levels=levels,
            colors='white',
            linewidths=lw,
            alpha=alpha,
            zorder=5
        )

    # --------------------------------------------
    # Main contours
    # --------------------------------------------

    contours = ax.contour(
        P,
        N,
        Z_contour,
        levels=levels,
        colors='#ffffffbb',
        linewidths=1.2,
        zorder=10
    )

    ax.clabel(
        contours,
        inline=True,
        fontsize=6,
        colors='white'
    )

    # ============================================
    # Zero contour
    # ============================================

    # Glow layers

    for lw, alpha in zip(
        [12, 9, 6],
        [0.02, 0.04, 0.08]
    ):

        ax.contour(
            P,
            N,
            Z_contour,
            levels=[0],
            colors=['#ffffffbb'],
            linewidths=lw,
            alpha=alpha,
            zorder=15
        )

    # Main zero contour

    zero_contour = ax.contour(
        P,
        N,
        Z_contour,
        levels=[0],
        colors=['#ffffffbb'],
        linewidths=1.2,
        zorder=20
    )

    ax.clabel(
        zero_contour,
        fmt='%d',
        fontsize=7,
        colors='white',
        manual=[(7, 1.5)]
    )

    # ============================================
    # Colorbar
    # ============================================

    cbar = plt.colorbar(
        mesh,
        ax=ax
    )

    cbar.ax.tick_params(
        labelsize=14,
        colors='white'
    )

    cbar.outline.set_edgecolor('white')
    cbar.outline.set_linewidth(2)

    cbar.set_label(
        r"$\Delta\Delta\Upsilon_n(\mathrm{pH})$ (kJ mol$^{-1}$)",
        fontsize=18,
        color='white'
    )

    # ============================================
    # Axes
    # ============================================

    ax.set_xlabel(
        'pH',
        fontsize=18
    )

    ax.set_ylabel(
        r'Fibril size ($n$)',
        fontsize=18
    )

    ax.set_yticks(n_values)

    ax.set_xlim(
        pHs.min(),
        pHs.max()
    )

    ax.set_ylim(0.5, 8.5)

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=14,
        length=6,
        width=2
    )

    ax.tick_params(
        axis='both',
        which='minor',
        length=3,
        width=1.5
    )

    ax.minorticks_on()
    ax.yaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')

    # ============================================
    # Final layout
    # ============================================

    fig.tight_layout()

    fig.savefig(
        'TEMP_POSTER/Heatmap_ddUpsilon_pH_0_norm_discrete_neon.pdf',
        bbox_inches='tight',
        transparent=True
    )


### Microscopic-Macrosopic mapping

In [24]:
# ==========================================================
# Imports
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Macroscopic reference
# ==========================================================

pK1_bar = 5.0
pK2_bar = 7.0

A = 10**pK1_bar + 10**pK2_bar
B = 10**(pK1_bar + pK2_bar)

# ==========================================================
# Compute W_max
# ==========================================================

W_max = np.log10((A**2) / (4*B))

# ==========================================================
# Sample W
# ==========================================================

W_vals = np.linspace(-4, W_max, 400)

pK1_branch1 = []
pK2_branch1 = []

pK1_branch2 = []
pK2_branch2 = []

W_plot = []

for W in W_vals:

    C = B * 10**W

    disc = A**2 - 4*C

    if disc < 0:
        continue

    sqrt_term = np.sqrt(disc)

    x1 = (A + sqrt_term) / 2
    x2 = (A - sqrt_term) / 2

    # Branch 1
    pK1_branch1.append(np.log10(x1))
    pK2_branch1.append(np.log10(x2))

    # Branch 2
    pK1_branch2.append(np.log10(x2))
    pK2_branch2.append(np.log10(x1))

    W_plot.append(W)

# ==========================================================
# Convert to arrays
# ==========================================================

pK1_branch1 = np.array(pK1_branch1)
pK2_branch1 = np.array(pK2_branch1)

pK1_branch2 = np.array(pK1_branch2)
pK2_branch2 = np.array(pK2_branch2)

W_plot = np.array(W_plot)

# ==========================================================
# Symmetry-breaking coordinates
# ==========================================================

delta1 = pK1_branch1 - pK2_branch1
delta2 = pK1_branch2 - pK2_branch2

# Symmetric endpoint

pK_sym = np.log10(A / 2)

# ==========================================================
# RC settings
# ==========================================================

with plt.rc_context({

    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': 'white',

    'figure.facecolor': '#FFFFFF',
    'axes.facecolor': '#FFFFFF',

    'axes.linewidth': 3,

    # Typography
    'font.family': 'Nimbus Sans',
    'font.weight': 'normal',

    'mathtext.fontset': 'stix',

    'axes.labelweight': 'normal',
    'axes.titleweight': 'normal',

    'text.usetex': False,

    # Slightly smaller figure
    'figure.figsize': (5.2, 3.5),

}):

    # ==========================================================
    # Neon palette
    # ==========================================================

    colors = ['#32ff14', '#ff7a00', '#32eefc', '#ee00ff']

    # ==========================================================
    # Figure
    # ==========================================================

    fig, ax = plt.subplots()

    ys = [delta1, delta2]

    plot_colors = [
        colors[1],  # cyan
        colors[2],  # magenta
    ]

    labels = [
        r'$\bar{\mathrm{p}}K_{a,1}=5,\ \bar{\mathrm{p}}K_{a,2}=7$',
        r'$\bar{\mathrm{p}}K_{a,1}=5,\ \bar{\mathrm{p}}K_{a,2}=7$'
    ]

    # ==========================================================
    # Neon glow settings
    # ==========================================================

    n_glow = 18
    glow_alpha = 0.035
    glow_width = 1.15

    # ==========================================================
    # Glow layers
    # ==========================================================

    for y, color in zip(ys, plot_colors):

        for n in range(1, n_glow+1):

            ax.plot(

                W_plot,
                y,

                color=color,

                linewidth=2 + glow_width*n,

                alpha=glow_alpha,

                solid_capstyle='round',

                zorder=1
            )

    # ==========================================================
    # Main foreground curves
    # ==========================================================

    for y, color, label in zip(ys, plot_colors, labels):

        ax.plot(

            W_plot,
            y,

            linewidth=2.5,

            color=color,

            label=label,

            solid_capstyle='round',

            zorder=10
        )

    # ==========================================================
    # Horizontal reference line
    # ==========================================================

    ax.axhline(

        0,

        linestyle='dashed',

        linewidth=2,

        color='white',

        alpha=0.35,

        zorder=0
    )

    # Glow for reference line

    for n in range(1, 10):

        ax.axhline(

            0,

            linewidth=2 + n*1.2,

            color='white',

            alpha=0.015,

            zorder=0
        )

    # ==========================================================
    # Symmetric endpoint
    # ==========================================================

    # Glow layers

    for n in range(1, 14):

        ax.scatter(

            W_max,
            0,

            s=40 + n*18,

            color=colors[0],

            alpha=0.025,

            zorder=15
        )

    # Main point

    ax.scatter(

        W_max,
        0,

        s=80,

        color=colors[0],

        zorder=20
    )

    # ==========================================================
    # Axes styling
    # ==========================================================

    ax.set_xlim(-0.1, 1.5)

    ax.set_ylim(-2.1, 2.1)

    ax.set_xlabel(
        r"$W$",
        fontsize=18
    )

    ax.set_ylabel(
        r"$\mathrm{p}K_{a,1}-\mathrm{p}K_{a,2}$",
        fontsize=18
    )

    ax.minorticks_on()

    ax.tick_params(

        axis='both',

        which='major',

        labelsize=14,

        length=6,

        width=2
    )

    ax.tick_params(

        axis='both',

        which='minor',

        length=3,

        width=1.5
    )

    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ==========================================================
    # Legend
    # ==========================================================

    ax.text(
        0.01,
        0.45,
        r'$\mathrm{p}\bar{K}_{a,1}=5$',
        fontsize=14,
        color='white'
    )
    
    ax.text(
        0.01,
        -0.65,
        r'$\mathrm{p}\bar{K}_{a,2}=7$',
        fontsize=14,
        color='white'
    )

    # ==========================================================
    # Final layout
    # ==========================================================

    fig.tight_layout()

    fig.savefig(

        'TEMP_POSTER/Symmetry_breaking_neon.pdf',

        bbox_inches='tight',

        transparent=True
    )

# ==========================================================
# Diagnostics
# ==========================================================

print("W_max =", W_max)
print("Symmetric point =", pK_sym)


W_max = 1.4065827562373228
Symmetric point = 6.703291378118662


### Microscopic titration curves

In [25]:
# ==========================================================
# Imports
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Data
# ==========================================================

pHs = np.array([
    0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5,
    4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0,
    7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 10.5,
    11.0, 11.5, 12.0, 12.5, 13.0, 13.5
])

y_minus1 = np.array([
    1.74166667e-04, 4.91666667e-04, 1.45416667e-03,
    4.01416667e-03, 9.89500000e-03, 2.07116667e-02,
    3.41783333e-02, 5.10941667e-02, 8.05233333e-02,
    1.43138333e-01, 2.67904167e-01, 4.58277500e-01,
    6.37474167e-01, 7.64145000e-01, 8.50012500e-01,
    9.19030000e-01, 9.66941667e-01, 9.88948333e-01,
    9.96474167e-01, 9.98733333e-01, 9.99594167e-01,
    9.99865000e-01, 9.99950000e-01, 9.99986667e-01,
    9.99995000e-01, 1.00000000e+00, 1.00000000e+00,
    1.00000000e+00
])

y_0 = np.array([
    0.0031875 , 0.01000417, 0.03094667, 0.08972833,
    0.22262583, 0.42933667, 0.62168333, 0.72911167,
    0.7709675 , 0.77267   , 0.7429625 , 0.6915575 ,
    0.66832417, 0.68705083, 0.75817417, 0.86400833,
    0.946045  , 0.9815975 , 0.9937825 , 0.99799083,
    0.99936333, 0.99985   , 0.999935  , 0.99997333,
    0.999995  , 0.99999083, 1.        , 1.
])

y_plus1 = np.array([
    5.85000000e-04, 1.49083333e-03, 4.36166667e-03,
    1.24075000e-02, 3.04066667e-02, 5.99575000e-02,
    8.93566667e-02, 1.11369167e-01, 1.33430000e-01,
    1.72570000e-01, 2.47831667e-01, 3.70817500e-01,
    4.94409167e-01, 6.02135000e-01, 7.06886667e-01,
    8.35839167e-01, 9.34658333e-01, 9.78606667e-01,
    9.93071667e-01, 9.97652500e-01, 9.99270000e-01,
    9.99772500e-01, 9.99923333e-01, 9.99988333e-01,
    1.00000000e+00, 1.00000000e+00, 1.00000000e+00,
    1.00000000e+00
])

# ==========================================================
# RC settings
# ==========================================================

with plt.rc_context({

    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': 'white',

    'figure.facecolor': '#FFFFFF',
    'axes.facecolor': '#FFFFFF',

    'axes.linewidth': 3,

    # Typography
    'font.family': 'Nimbus Sans',
    'font.weight': 'normal',

    'mathtext.fontset': 'stix',

    'axes.labelweight': 'normal',
    'axes.titleweight': 'normal',

    'text.usetex': False,

    'figure.figsize': (6, 4),

}):

    # ==========================================================
    # Neon palette
    # ==========================================================

    colors = ['#32ff14', '#ff7a00', '#32eefc', '#ee00ff']

    ys = [y_minus1, y_0, y_plus1]

    plot_colors = [
        colors[0],  # cyan
        colors[1],  # orange
        colors[2],  # magenta
    ]

    labels = [
        r'$-1$',
        r'$0$',
        r'$+1$'
    ]

    # ==========================================================
    # Figure
    # ==========================================================

    fig, ax = plt.subplots()

    # ==========================================================
    # Neon glow settings
    # ==========================================================

    n_glow = 18
    glow_alpha = 0.035
    glow_width = 1.15

    # ==========================================================
    # Glow layers
    # ==========================================================

    for y, color in zip(ys, plot_colors):

        for n in range(1, n_glow+1):

            ax.plot(

                pHs,
                y,

                color=color,

                linewidth=2 + glow_width*n,

                alpha=glow_alpha,

                solid_capstyle='round',

                zorder=1
            )

    # ==========================================================
    # Main foreground curves
    # ==========================================================

    for y, color, label in zip(ys, plot_colors, labels):

        ax.plot(

            pHs,
            y,

            linewidth=2.5,

            marker='o',
            markersize=5,

            color=color,

            label=label,

            solid_capstyle='round',

            zorder=10
        )

    # ==========================================================
    # Axes styling
    # ==========================================================

    ax.set_xlabel(
        'pH',
        fontsize=18
    )

    ax.set_ylabel(
        'Deprotonated fraction',
        fontsize=18
    )

    ax.set_xlim(0, 10)

    ax.set_ylim(-0.05, 1.05)

    ax.minorticks_on()

    ax.tick_params(

        axis='both',

        which='major',

        labelsize=14,

        length=6,

        width=2
    )

    ax.tick_params(

        axis='both',

        which='minor',

        length=3,

        width=1.5
    )

    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ==========================================================
    # Legend
    # ==========================================================

    legend = ax.legend(

        frameon=False,

        fontsize=14,

        loc='best'
    )

    for text in legend.get_texts():
        text.set_color('white')

    # ==========================================================
    # Final layout
    # ==========================================================

    fig.tight_layout()

    fig.savefig(

        'TEMP_POSTER/Glu_B13_titration_neon.pdf',

        bbox_inches='tight',

        transparent=True
    )

### Contacts 

In [26]:
# ==========================================================
# Data
# ==========================================================

pHs = np.array([
    0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5,
    4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0,
    7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 10.5,
    11.0, 11.5, 12.0, 12.5, 13.0, 13.5
])

# ==========================================================
# Contact data
# ==========================================================

C_tot_norm = np.array([
    1., 1.00029472, 1.00033204, 1.00117307, 1.00309921,
    1.00642985, 1.00965739, 1.01318923, 1.01555883,
    1.0178837, 1.01755579, 1.0151578, 1.01089873,
    1.00356967, 0.99381196, 0.98353526, 0.97596788,
    0.97288672, 0.97130887, 0.97069726, 0.96851873,
    0.96382738, 0.94934342, 0.9201835, 0.87764994,
    0.84106749, 0.81511589, 0.79566919
])

C_tot_err_norm = np.array([
    0.00460034, 0.00457734, 0.00462594, 0.00478645,
    0.00467757, 0.00503146, 0.0056491, 0.00581365,
    0.00587377, 0.00595869, 0.00556292, 0.00550012,
    0.00553197, 0.00512929, 0.00456154, 0.00406308,
    0.00341475, 0.00324468, 0.00314875, 0.00324152,
    0.00329436, 0.00326415, 0.0036098, 0.00423062,
    0.00505055, 0.00505406, 0.00508491, 0.00516388
])

C_glu_glu_norm = np.array([
    1., 1.00264529, 1.00373832, 1.01175405, 1.02821499,
    1.05767176, 1.08716168, 1.11803803, 1.14261113,
    1.1570628, 1.14193249, 1.10369849, 1.04977439,
    0.98395532, 0.90584609, 0.82599522, 0.77407291,
    0.75372639, 0.74688728, 0.7442998, 0.74365841,
    0.74240753, 0.74087557, 0.73801922, 0.73467162,
    0.73275294, 0.73152137, 0.7306516
])

C_glu_glu_err_norm = np.array([
    0.00809377, 0.00848525, 0.00824055, 0.00839878,
    0.00985552, 0.0101854, 0.01053997, 0.01124131,
    0.01226557, 0.0127108, 0.01210246, 0.01295599,
    0.01362415, 0.01283059, 0.01242142, 0.00864395,
    0.0047089, 0.00273535, 0.00213149, 0.00206959,
    0.00218087, 0.00220191, 0.00211029, 0.0020789,
    0.00234736, 0.00348155, 0.00368734, 0.00474411
])

C_glu_other_norm = np.array([
    1., 0.99994022, 0.99816942, 0.99609842, 0.9890028,
    0.97852139, 0.96593828, 0.95270937, 0.93872033,
    0.92674708, 0.92070852, 0.91594391, 0.91535887,
    0.91695557, 0.91065143, 0.89167828, 0.87221122,
    0.86423166, 0.8592972, 0.85828065, 0.85729186,
    0.85669931, 0.85348076, 0.84900068, 0.84684736,
    0.85212645, 0.86015141, 0.86368942
])

C_glu_other_err_norm = np.array([
    0.00892024, 0.00910527, 0.00888461, 0.00867831,
    0.00823666, 0.00840875, 0.00917617, 0.00997637,
    0.01015746, 0.01125226, 0.01234008, 0.01451096,
    0.0155577, 0.01361096, 0.0112742, 0.00855892,
    0.00747317, 0.00835329, 0.00834857, 0.00868636,
    0.0086946, 0.00839413, 0.00897045, 0.01008211,
    0.01166127, 0.01383095, 0.01557846, 0.01780558
])

C_other_other_norm = np.array([
    1., 0.99988432, 0.99985383, 0.99957152, 0.99944255,
    0.99890926, 0.99838885, 0.99803942, 0.99745169,
    0.99854033, 1.00135803, 1.00576581, 1.0104475,
    1.01332295, 1.01598323, 1.01920947, 1.02069954,
    1.02116285, 1.0207968, 1.02057983, 1.01803248,
    1.0124142, 0.99473493, 0.95895856, 0.90631278,
    0.86035119, 0.82741962, 0.80290757
])

C_other_other_err_norm = np.array([
    0.00617829, 0.0061367, 0.00618307, 0.00647447,
    0.00639183, 0.00676819, 0.00753912, 0.00788505,
    0.0080717, 0.00812757, 0.0074174, 0.00675917,
    0.00640912, 0.00608363, 0.00546308, 0.00475065,
    0.00401503, 0.00393071, 0.00392404, 0.00403122,
    0.00410168, 0.00401333, 0.00439054, 0.00506581,
    0.00603579, 0.005925, 0.00586689, 0.00595902
])

# ==========================================================
# RC settings
# ==========================================================

with plt.rc_context({

    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': 'white',

    'figure.facecolor': '#FFFFFF',
    'axes.facecolor': '#FFFFFF',

    'axes.linewidth': 3,

    # Typography
    'font.family': 'Nimbus Sans',
    'font.weight': 'normal',

    'mathtext.fontset': 'stix',

    'axes.labelweight': 'normal',
    'axes.titleweight': 'normal',

    'text.usetex': False,

    'figure.figsize': (6, 4),

}):

    # ==========================================================
    # Neon palette
    # ==========================================================

    colors = ['#32ff14', '#ff7a00', '#32eefc', '#ee00ff']

    curves = [

        (
            C_tot_norm,
            C_tot_err_norm,
            colors[0],
            'o',
            'Total contacts'
        ),

        (
            C_glu_glu_norm,
            C_glu_glu_err_norm,
            colors[1],
            'o',
            'Glu–Glu'
        ),

        (
            C_glu_other_norm,
            C_glu_other_err_norm,
            colors[2],
            'o',
            'Glu–Other'
        ),

        (
            C_other_other_norm,
            C_other_other_err_norm,
            colors[3],
            'o',
            'Other–Other'
        ),
    ]

    # ==========================================================
    # Figure
    # ==========================================================

    fig, ax = plt.subplots()

    # ==========================================================
    # Glow settings
    # ==========================================================

    n_glow = 18
    glow_alpha = 0.03
    glow_width = 1.05

    # ==========================================================
    # Plot curves
    # ==========================================================

    for y, err, color, marker, label in curves:

        # ------------------------------------------------------
        # Error band
        # ------------------------------------------------------

        ax.fill_between(

            pHs,

            y - err,
            y + err,

            color=color,

            alpha=0.14,

            zorder=1
        )

        # ------------------------------------------------------
        # Glow layers
        # ------------------------------------------------------

        for n in range(1, n_glow+1):

            ax.plot(

                pHs,
                y,

                color=color,

                linewidth=2 + glow_width*n,

                alpha=glow_alpha,

                solid_capstyle='round',

                zorder=2
            )

        # ------------------------------------------------------
        # Main foreground curve
        # ------------------------------------------------------

        ax.plot(

            pHs,
            y,

            linewidth=2.5,

            marker=marker,
            markersize=5,

            color=color,

            label=label,

            zorder=10
        )

    # ==========================================================
    # Reference line
    # ==========================================================

    ax.axhline(

        1.0,

        color='white',

        linestyle='dashed',

        linewidth=2,

        alpha=0.35,

        zorder=0
    )

    for n in range(1, 10):

        ax.axhline(

            1.0,

            linewidth=2 + n*1.2,

            color='white',

            alpha=0.015,

            zorder=0
        )

    # ==========================================================
    # Axes styling
    # ==========================================================

    ax.set_xlabel(
        'pH',
        fontsize=18
    )

    ax.set_ylabel(
        'Inter-monomer sidechain\ncontacts (normalized to pH 0)',
        fontsize=18
    )

    ax.set_xlim(0, 13.5)

    ax.minorticks_on()

    ax.tick_params(

        axis='both',

        which='major',

        labelsize=14,

        length=6,

        width=2
    )

    ax.tick_params(

        axis='both',

        which='minor',

        length=3,

        width=1.5
    )

    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ==========================================================
    # Legend
    # ==========================================================

    legend = ax.legend(

        frameon=False,

        fontsize=13,

        loc='best'
    )

    for text in legend.get_texts():
        text.set_color('white')

    # ==========================================================
    # Layout + save
    # ==========================================================

    fig.tight_layout()

    fig.savefig(

        'TEMP_POSTER/Fibril_3mer_contacts_neon.pdf',

        bbox_inches='tight',

        transparent=True
    )
